##### Installation

In [4]:
! pip install sentence-transformers faiss-cpu pandas numpy

'pip' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


In [2]:
! pip install tensorflow



##### Charger le modèle d’embeddings

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


ModuleNotFoundError: No module named 'sentence_transformers'

##### Préparer les données Amazon Q&A

In [6]:
import ast
import pandas as pd

rows = []

with open("qa_Baby.json", "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rows.append(ast.literal_eval(line))

data = pd.DataFrame(rows)

print(data.shape)
print(data.columns)
print(data.head(2))


(28933, 7)
Index(['questionType', 'asin', 'answerTime', 'unixTime', 'question', 'answer',
       'answerType'],
      dtype='object')
  questionType        asin    answerTime      unixTime  \
0   open-ended  177036417X  Apr 16, 2015  1.429168e+09   
1       yes/no  177036417X   Jul 1, 2014  1.404198e+09   

                                            question  \
0  Does this book contain any vaccination/immuniz...   
1  Does this book have a section for the baby sho...   

                                answer answerType  
0  Immunization page, yes. School, no.        NaN  
1                          Yes it does          Y  


##### Ajout de la catégorie et Nettoyage minimal

In [7]:
data = data.dropna(subset=["question", "answer"])
data["category"] = "Baby"

##### Création du champ text

In [8]:
data["text"] = (
    "product category: baby. "
    "question: " + data["question"].astype(str) +
    " answer: " + data["answer"].astype(str)
).str.lower()


##### Vérification

In [9]:
print(data[["asin", "question", "answer", "text"]].head(2))


         asin                                           question  \
0  177036417X  Does this book contain any vaccination/immuniz...   
1  177036417X  Does this book have a section for the baby sho...   

                                answer  \
0  Immunization page, yes. School, no.   
1                          Yes it does   

                                                text  
0  product category: baby. question: does this bo...  
1  product category: baby. question: does this bo...  


##### Embeddings des documents

In [8]:
doc_embeddings = model.encode(
    data["text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)


NameError: name 'model' is not defined

##### Construction de l’index

In [9]:
import numpy as np

dim = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dim)
index.add(doc_embeddings)

print("Nombre de vecteurs indexés :", index.ntotal)


NameError: name 'doc_embeddings' is not defined

##### Fonction de recherche

In [ ]:
def retrieve(query, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)
    return data.iloc[indices[0]]


Test réel

In [ ]:
query = "Is this baby book religious?"
results = retrieve(query)

results[["question", "answer"]]


In [ ]:
def build_prompt(query, retrieved_docs):
    context = "\n".join(
        "- " + row["answer"]
        for _, row in retrieved_docs.iterrows()
    )

    prompt = f"""
You are a helpful e-commerce assistant.

Context:
{context}

User question:
{query}

Answer concisely and accurately.
"""
    return prompt


In [ ]:
prompt = build_prompt(query, results)
print(prompt)
